# Vision Evaluator

Evaluates a **vision-language model** on a set of local images.

For each (image, prompt) pair the evaluator:
- Base64-encodes the image and attaches it to a chat request
- Records latency, token counts and the model's description

Images are loaded from a local folder and sorted by file size, giving a natural sweep from small to large images.

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelVisionEvaluator

## 2. Prepare sample images

The repo ships sample images under `resources/data/images/`. You can also download your own using the helper below.
Skip this cell if you already have images in your folder.

In [ ]:
import os

# Use bundled sample images
IMAGE_FOLDER = "/workspaces/BeLLMira/resources/data/images"
print(f"Images found: {[f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith(('.png','.jpg','.jpeg'))]}")

### Optional: download images from Hugging Face

In [ ]:
# Uncomment to download images from HF URLs
#
# from bellmira.evaluators.model_vision_evaluator import download_images_from_huggingface
#
# urls = [
#     "https://huggingface.co/datasets/your-dataset/resolve/main/image1.png",
#     "https://huggingface.co/datasets/your-dataset/resolve/main/image2.png",
# ]
# saved = download_images_from_huggingface(urls, local_path="/tmp/vision_images", hf_token="hf_...")
# IMAGE_FOLDER = "/tmp/vision_images"

## 3. Configuration

In [ ]:
# Must be a vision-language model (e.g. Qwen2.5-VL)
MODEL_URL = "https://chatqwen25vl-backend.dat-aip-millm.qa.mbcp.cloud/"

PROMPTS = [
    "Identify the elements in the image and describe them in a list.",
    "What is the main subject of this image?",
]

## 4. Initialise the evaluator

In [ ]:
evaluator = ModelVisionEvaluator(
    url=MODEL_URL,
    image_folder_path=IMAGE_FOLDER,
    prompts=PROMPTS,
    temperature=0.0,
)

print(f"Model: {evaluator.model_name}")
print(f"Images loaded ({len(evaluator.images)}):")
for key in evaluator.images:
    print(f"  {key}")

## 5. Run evaluation

In [ ]:
# max_prompts=1 means one prompt per image
results = evaluator.evaluate(max_prompts=1)

for image_key, runs in results.items():
    print(f"\n--- {image_key} ---")
    for r in runs:
        print(f"  latency={r.get('Execution_time', 'ERR'):.2f}s  "
              f"tokens={r.get('Total_tokens')}")

## 6. Compute averages and extract metrics

In [ ]:
averages = evaluator.compute_averages(results)
for key, avg in averages.items():
    print(f"{key}: avg_exec={avg.get('Avg_execution_time')}s  "
          f"avg_prompt_tok={avg.get('Avg_prompt_tokens')}  "
          f"avg_compl_tok={avg.get('Avg_completion_tokens')}")

In [ ]:
metrics = evaluator.extract_threshold_metrics(
    results,
    metrics=["Avg_execution_time", "Avg_completion_tokens"],
    key_mode="dim",
)
for key, val in metrics.items():
    print(f"{key}: {val}")

## 7. Inspect model responses

In [ ]:
# Re-run a single image with streaming to see the response live
from bellmira.llm_model.llm_model_client import ModelClient
import base64

client = ModelClient(base_url=MODEL_URL)
model_name = client.get_model_name()

# Pick the first image
first_image_path = list(evaluator.images.values())[0]
with open(first_image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

req = client.build_chat_request(
    user_prompt="Describe this image in detail.",
    system_prompt="Analyse the next image and tell what it is about.",
    model_name=model_name,
    temperature=0.7,
    image_prompt=image_b64,
)

for token in client.stream_chat_response(req):
    print(token, end="", flush=True)